In [ ]:
# %%
import time
import os
import pandas as pd
import random
import json
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    run_all_pairwise_projections_parallel,
    generate_pairwise_comparisons_from_df
)

# %%
# --- Pfade & Daten einlesen ---
data_path = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits/eurasian_otter_(1)")
df = pd.read_csv(data_path / "train.csv")

# %%
# --- Feature-Spalten auswählen ---
feature_cols = [
    col for col in df.columns
    if col.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[col])
]
print(f"{len(feature_cols)} numerische Features gefunden.")

# %%
# --- Gemeinsame Parameter ---
feature_counts    = [5,10,16,100]
selection_methods = ["random_forest","forward"]
reducers          = ["pca","lda","umap"]
n_components_list = [2,3   ]   # nur 2n_components_list = [2]   # nur 2
use_sexmodel_prediction: [True, False] = [True, False
                                          ]
n_jobs            = -1

output_dir = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data")
output_dir.mkdir(parents=True, exist_ok=True)

# %%
# --- Vergleiche generieren & downsamplen auf 500 Pairs ---
comparisons = generate_pairwise_comparisons_from_df(
    df,
    group_sizes=[2,3,4,5,6,7,8,9,10],
    n_repeats=2,
    mode="both",
    selfmatch_factor=10
)
comparisons = random.sample(comparisons, 5000)

# %%
all_results = []
timing_log  = []

# %%
# --- Für beide Settings: Sexmodell off/on ---
for sel in tqdm(selection_methods, desc=f"FS ({tag})"):
    t0 = time.time()
    with tqdm_joblib(tqdm(desc=f"Pairs ({sel}, {tag})", total=len(comparisons))):
        results = run_all_pairwise_projections_parallel(
            comparisons=comparisons,
            df=df,
            feature_cols=feature_cols,
            k_features=feature_counts,
            reducers=reducers,
            selection_method=sel,
            n_components=n_components_list,
            debug=True,
            use_sexmodel_prediction=use_sex,
            sexmodel_path=(
            r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\models\eurasian_otter_(1).joblib"
                    if use_sex else None
                ),
                n_jobs=n_jobs
            )
        dur = time.time() - t0
        print(f"{sel} | sex={use_sex}: {len(results)} Ergebnisse in {dur:.1f}s")
        print("Sample:", results[:2])

        df_res = pd.DataFrame(results)
        df_res["selection_method"] = sel
        df_res["use_sexmodel_prediction"] = use_sex
        fname = f"pairwise_{sel}_{tag}_{len(comparisons)}pairs.csv"
        df_res.to_csv(output_dir / fname, index=False)

        all_results.append(df_res)
        timing_log.append({
            "selection_method": sel,
            "use_sexmodel_prediction": use_sex,
            "duration_s": dur,
            "n_results": len(results)
        })

# %%
# --- Timing & Gesamtergebnis ---
pd.DataFrame(timing_log).to_csv(output_dir / "timing_log.csv", index=False)

if all_results:
    full = pd.concat(all_results, ignore_index=True)
    full.to_csv(output_dir / "all_results_combined.csv", index=False)
    display(full.head())
else:
    print("Keine Ergebnisse zum Zusammenführen.")


205 numerische Features gefunden.


FS (sex_off):   0%|          | 0/2 [00:00<?, ?it/s]

Pairs (random_forest, sex_off):   0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
# %%
import time
import os
import pandas as pd
import random
import json
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    run_all_pairwise_projections_parallel,
    generate_pairwise_comparisons_from_df
)

# %%
# --- Pfade & Daten einlesen ---
data_path = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits/eurasian_otter_(1)")
df = pd.read_csv(data_path / "train.csv")

# %%
# --- Feature-Spalten auswählen ---
feature_cols = [
    col for col in df.columns
    if col.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[col])
]
print(f"{len(feature_cols)} numerische Features gefunden.")

# %%
# --- Parameter definieren ---
feature_counts    = [2,5,8,10,12,14,16,18,20,30,50,100,150,200]     # k = 2…19
reducers          = ["pca","lda","umap"]
selection_methods = ["forward","random_forest"]
n_components_list = list(range(2, 6))       # Komponenten = 2…5
n_jobs            = 1 # Parallelisierung, 1 für Debugging -1 max speed


output_dir = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results_long/data")
output_dir.mkdir(parents=True, exist_ok=True)

# %%
# --- Vergleiche einmal generieren & downsamplen auf 10 Pairs ---
comparisons = generate_pairwise_comparisons_from_df(
    df,
    group_sizes=[3,4,5,6,7,8],
    n_repeats=5,
    mode="both",
    selfmatch_factor=20
)
#comparisons = random.sample(comparisons, 10)
#print("Erste 3 Paare:", comparisons[:3])

# %%
all_results = []
timing_log  = []

# %%
# --- Sweep über selection_method mit Fortschrittsanzeigen ---
for sel in tqdm(selection_methods, desc="Selection methods"):
    t0 = time.time()
    # innere Bar für die Pairs
    with tqdm_joblib(tqdm(desc=f"Pairs ({sel})", total=len(comparisons))):
        results = run_all_pairwise_projections_parallel(
            comparisons=comparisons,
            df=df,
            feature_cols=feature_cols,
            k_features=feature_counts,       # komplette Liste k
            reducers=reducers,               # alle Reducer
            selection_method=sel,
            n_components=n_components_list,  # alle NCs
            debug=True,
            n_jobs=n_jobs
        )
    dur = time.time() - t0
    print(f"{sel}: {len(results)} Ergebnisse in {dur:.1f}s")
    # optional: erstes Ergebnis anschauen
    print("Sample:", results[:2])

    # In DataFrame und abspeichern
    df_res = pd.DataFrame(results)
    df_res["selection_method"] = sel
    fname = f"pairwise_{sel}_{len(comparisons)}pairs.csv"
    df_res.to_csv(output_dir / fname, index=False)

    all_results.append(df_res)
    timing_log.append({"selection_method": sel, "duration_s": dur, "n_results": len(results)})

# %%
# --- Timing & Gesamt-CSV ---
pd.DataFrame(timing_log).to_csv(output_dir / "timing_log.csv", index=False)

if all_results:
    full = pd.concat(all_results, ignore_index=True)
    full.to_csv(output_dir / "all_results_combined.csv", index=False)
    display(full.head())
else:
    print("Keine Ergebnisse zum Zusammenführen.")


Processing Pairs:   0%|          | 0/100 [00:27<?, ?it/s]

205 numerische Features gefunden.



Processing Pairs:   0%|          | 0/100 [00:10<?, ?it/s]


Selection methods:   0%|          | 0/2 [00:00<?, ?it/s]

Pairs (forward):   0%|          | 0/113590 [00:00<?, ?it/s]

  0%|          | 0/113590 [00:00<?, ?it/s]

  0%|          | 0/113590 [00:00<?, ?it/s]





































































































































































c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(






c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensur

In [ ]:
# nimm das erste Pair
single_pair = [comparisons[0]]

# ein einzelner Durchlauf, debug=True, n_jobs=1
res = run_all_pairwise_projections_parallel(
    comparisons     = single_pair,
    df              = df,
    feature_cols    = feature_cols,
    k_features      = [5, 10],       # z.B. nur zwei k-Werte, damit's übersichtlich bleibt
    reducers        = ["pca"],       # erstmal nur PCA
    selection_method= "forward",     # oder "random_forest"
    n_components    = [2],           # nur 2 Komponenten
    debug           = True,
    n_jobs          = 1
)

print("Result for 1 pair:", res)


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    balanced_accuracy_score
)
import matplotlib.pyplot as plt

# Lade kombiniertes Ergebnis
df_model = pd.read_csv(os.path.join(output_dir, "all_results_combined.csv"))

# Leere Liste für Ergebnis-Log
model_scores = []

# Iteriere über alle k-Werte
for k in sorted(df_model["k_features"].unique()):
    df_k = df_model[df_model["k_features"] == k]

    X = df_k[[
        "center_distance_ab",
        "center_distance_rcv_a",
        "center_distance_rcv_b"
    ]]
    y = df_k["same_individual"].astype(int)

    if y.nunique() < 2:
        print(f"Skipping k={k} – nur eine Klasse vorhanden.")
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_prob)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print(f"\n=== k = {k} ===")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc:.3f}")
    print(f"Balanced Accuracy: {bal_acc:.3f}")

    model_scores.append({
        "k_features": k,
        "roc_auc": roc_auc,
        "balanced_accuracy": bal_acc
    })

# Zusammenfassen und plotten
scores_df = pd.DataFrame(model_scores)

plt.figure(figsize=(10, 5))
plt.plot(scores_df["k_features"], scores_df["roc_auc"], label="ROC-AUC", marker="o")
plt.plot(scores_df["k_features"], scores_df["balanced_accuracy"], label="Balanced Accuracy", marker="s")
plt.xlabel("Anzahl ausgewählter Features (k)")
plt.ylabel("Score")
plt.title("Modellperformance vs. Anzahl Features")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
